# Conjunto de datos completo sin clusterización

In [1]:
#Importaciones
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

#Lectura de datos
datos = pd.read_excel('03_Clusterizacion_CTNET.xlsx')
datos.head(24)

,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM
0,2022-09-01 00:00:00,0.000000,19,77,0,0,Noche,Noche
1,2022-09-01 01:00:00,0.000000,19,82,0,1,Noche,Noche
2,2022-09-01 02:00:00,0.000000,18,85,0,2,Noche,Noche
3,2022-09-01 03:00:00,0.000000,18,87,0,3,Noche,Noche
4,2022-09-01 04:00:00,0.000000,18,88,0,4,Noche,Noche
5,2022-09-01 05:00:00,0.000000,17,86,0,5,Noche,Noche
6,2022-09-01 06:00:00,0.000000,18,89,0,6,Soleado,Lluvioso
7,2022-09-01 07:00:00,6.584959,18,95,0,7,Soleado,Lluvioso
8,2022-09-01 08:00:00,560.422022,18,100,0,8,Soleado,Lluvioso
9,2022-09-01 09:00:00,7720.582326,18,100,1,9,Soleado,Lluvioso


In [2]:
datos["Generacion_prev_hour"] = datos["Generación"].shift(1)
datos["Generacion_prev_day"] = datos["Generación"].shift(24)
datos = datos.dropna(how="any", axis= 0)

Definimos X y y

In [3]:
datos_dia = datos[datos["Cluster KMeans"] == "Soleado"].copy()
datos_dia.head(10)

,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day
30,2022-09-02 06:00:00,0.000000,17,91,0,6,Soleado,Lluvioso,0.000000,0.000000
31,2022-09-02 07:00:00,0.000000,17,94,0,7,Soleado,Lluvioso,0.000000,6.584959
32,2022-09-02 08:00:00,438.814997,16,97,0,8,Soleado,Lluvioso,0.000000,560.422022
33,2022-09-02 09:00:00,5908.000884,17,93,1,9,Soleado,Lluvioso,438.814997,7720.582326
34,2022-09-02 10:00:00,5030.740421,18,85,2,10,Soleado,Lluvioso,5908.000884,9433.109309
35,2022-09-02 11:00:00,17036.043251,20,71,2,11,Soleado,Lluvioso,5030.740421,22189.147406
44,2022-09-02 20:00:00,2370.417542,24,52,0,20,Soleado,Lluvioso,11972.590689,3411.083740
45,2022-09-02 21:00:00,182.435112,22,61,0,21,Soleado,Lluvioso,2370.417542,382.187718
54,2022-09-03 06:00:00,0.000000,18,87,0,6,Soleado,Lluvioso,0.000000,0.000000
55,2022-09-03 07:00:00,13.512200,18,87,0,7,Soleado,Lluvioso,0.000000,0.000000


In [4]:
columns = datos_dia.drop(columns=["Fecha", "Generación", "Cluster KMeans", "Cluster GMM"]).columns

In [5]:
X = datos_dia[columns]
X

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
30,17,91,0,6,0.000000,0.000000
31,17,94,0,7,0.000000,6.584959
32,16,97,0,8,0.000000,560.422022
33,17,93,1,9,438.814997,7720.582326
34,18,85,2,10,5908.000884,9433.109309
...,...,...,...,...,...,...
18273,14,87,1,8,67.000000,7302.000000
18274,15,83,2,9,7356.000000,18014.000000
18275,17,71,4,10,17638.000000,23010.000000
18276,19,60,5,11,23339.000000,26156.000000


In [6]:
y = datos_dia[['Generación']]
y

,Generación
30,0.000000
31,0.000000
32,438.814997
33,5908.000884
34,5030.740421
...,...
18273,7356.000000
18274,17638.000000
18275,23339.000000
18276,26323.000000


Dividimos entrenamiento, validación y prueba

In [7]:
train_size = int(0.7 * len(X))
val_size = int(0.85 * len(X))

In [8]:
# Entrenamiento, validación y prueba, 75, 15 y 15
X_train, y_train =  X.iloc[:train_size, :], y.iloc[:train_size, :]
X_val, y_val = X.iloc[train_size:val_size, :], y.iloc[train_size:val_size, :]
X_test, y_test = X.iloc[val_size:, :],  y.iloc[val_size:,:]

print(f'X_train: {len(X_train)}, y_train: {len(y_train)}')
print(f'X_val: {len(X_val)}, y_val: {len(y_val)}')
print(f'X_test: {len(X_test)}, y_test: {len(y_test)}')

X_train: 3882, y_train: 3882
X_val: 832, y_val: 832
X_test: 832, y_test: 832


## Escalar con MinMaxScaler

In [9]:
from sklearn.preprocessing import MinMaxScaler

In [10]:
x_scaler = MinMaxScaler().fit(X_train)
x_scaler

MinMaxScaler()

In [11]:
X_train_scaled = x_scaler.transform(X_train)
print(X_train_scaled)
print(X_train_scaled.shape)

[[6.53846154e-01 8.67647059e-01 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [6.53846154e-01 9.11764706e-01 0.00000000e+00 6.66666667e-02
  0.00000000e+00 2.19498623e-04]
 [6.15384615e-01 9.55882353e-01 0.00000000e+00 1.33333333e-01
  0.00000000e+00 1.86807341e-02]
 ...
 [4.61538462e-01 6.47058824e-01 2.85714286e-01 2.66666667e-01
  2.05400000e-01 5.44266667e-01]
 [6.15384615e-01 3.82352941e-01 2.85714286e-01 3.33333333e-01
  5.85333333e-01 6.35166667e-01]
 [3.84615385e-01 7.20588235e-01 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00]]
(3882, 6)


In [12]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_train_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
30,0.653846,0.867647,0.000000,0.000000,0.000000,0.000000
31,0.653846,0.911765,0.000000,0.066667,0.000000,0.000219
32,0.615385,0.955882,0.000000,0.133333,0.000000,0.018681
33,0.653846,0.897059,0.142857,0.200000,0.014627,0.257353
34,0.692308,0.779412,0.285714,0.266667,0.196933,0.314437
...,...,...,...,...,...,...
12201,0.269231,0.985294,0.000000,0.133333,0.000000,0.000867
12202,0.346154,0.852941,0.142857,0.200000,0.002433,0.050100
12203,0.461538,0.647059,0.285714,0.266667,0.205400,0.544267
12204,0.615385,0.382353,0.285714,0.333333,0.585333,0.635167


In [13]:
X_val_scaled = x_scaler.transform(X_val)
print(X_val_scaled)
print(X_val_scaled.shape)

[[0.38461538 0.76470588 0.         0.06666667 0.         0.        ]
 [0.30769231 0.86764706 0.         0.13333333 0.         0.00243333]
 [0.38461538 0.75       0.14285714 0.2        0.00126667 0.2054    ]
 ...
 [0.76923077 0.64705882 0.57142857 0.26666667 0.82916667 0.91103333]
 [0.92307692 0.38235294 0.14285714 0.86666667 0.6946     0.19466667]
 [0.88461538 0.45588235 0.         0.93333333 0.36236667 0.0164    ]]
(832, 6)


In [14]:
X_val_scaled_df = pd.DataFrame(X_val_scaled, index=X_val.index, columns=X_val.columns)
X_val_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
12224,0.384615,0.764706,0.000000,0.066667,0.000000,0.000000
12225,0.307692,0.867647,0.000000,0.133333,0.000000,0.002433
12226,0.384615,0.750000,0.142857,0.200000,0.001267,0.205400
12227,0.500000,0.558824,0.142857,0.266667,0.083400,0.585333
12228,0.615385,0.352941,0.285714,0.333333,0.585333,0.635167
...,...,...,...,...,...,...
15897,0.692308,0.867647,0.285714,0.133333,0.077467,0.516633
15898,0.730769,0.779412,0.428571,0.200000,0.517867,0.824900
15899,0.769231,0.647059,0.571429,0.266667,0.829167,0.911033
15908,0.923077,0.382353,0.142857,0.866667,0.694600,0.194667


In [15]:
X_test_scaled = x_scaler.transform(X_test)
print(X_test_scaled)
print(X_test_scaled.shape)

[[0.84615385 0.57352941 0.         1.         0.04526667 0.        ]
 [0.65384615 0.92647059 0.         0.         0.         0.        ]
 [0.61538462 1.         0.         0.06666667 0.         0.07746667]
 ...
 [0.65384615 0.57352941 0.57142857 0.26666667 0.58793333 0.767     ]
 [0.73076923 0.41176471 0.71428571 0.33333333 0.77796667 0.87186667]
 [0.76923077 0.32352941 0.         1.         0.         0.        ]]
(832, 6)


In [16]:
X_test_scaled_df = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
X_test_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
15910,0.846154,0.573529,0.000000,1.000000,0.045267,0.000000
15919,0.653846,0.926471,0.000000,0.000000,0.000000,0.000000
15920,0.615385,1.000000,0.000000,0.066667,0.000000,0.077467
15921,0.653846,0.882353,0.285714,0.133333,0.077467,0.517867
15922,0.692308,0.779412,0.428571,0.200000,0.546133,0.829167
...,...,...,...,...,...,...
18273,0.538462,0.808824,0.142857,0.133333,0.002233,0.243400
18274,0.576923,0.750000,0.285714,0.200000,0.245200,0.600467
18275,0.653846,0.573529,0.571429,0.266667,0.587933,0.767000
18276,0.730769,0.411765,0.714286,0.333333,0.777967,0.871867


In [17]:
x_scaller_all = MinMaxScaler().fit(X)
print(x_scaller_all)

MinMaxScaler()


In [18]:
X_scaled = x_scaller_all.transform(X)
print(X_scaled)
print(X_scaled.shape)

[[6.53846154e-01 8.75000000e-01 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [6.53846154e-01 9.16666667e-01 0.00000000e+00 6.66666667e-02
  0.00000000e+00 2.19498623e-04]
 [6.15384615e-01 9.58333333e-01 0.00000000e+00 1.33333333e-01
  0.00000000e+00 1.86807341e-02]
 ...
 [6.53846154e-01 5.97222222e-01 4.44444444e-01 2.66666667e-01
  5.87933333e-01 7.67000000e-01]
 [7.30769231e-01 4.44444444e-01 5.55555556e-01 3.33333333e-01
  7.77966667e-01 8.71866667e-01]
 [7.69230769e-01 3.61111111e-01 0.00000000e+00 1.00000000e+00
  0.00000000e+00 0.00000000e+00]]
(5546, 6)


In [19]:
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)
X_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
30,0.653846,0.875000,0.000000,0.000000,0.000000,0.000000
31,0.653846,0.916667,0.000000,0.066667,0.000000,0.000219
32,0.615385,0.958333,0.000000,0.133333,0.000000,0.018681
33,0.653846,0.902778,0.111111,0.200000,0.014627,0.257353
34,0.692308,0.791667,0.222222,0.266667,0.196933,0.314437
...,...,...,...,...,...,...
18273,0.538462,0.819444,0.111111,0.133333,0.002233,0.243400
18274,0.576923,0.763889,0.222222,0.200000,0.245200,0.600467
18275,0.653846,0.597222,0.444444,0.266667,0.587933,0.767000
18276,0.730769,0.444444,0.555556,0.333333,0.777967,0.871867


In [20]:
y_scaler = MinMaxScaler().fit(y_train)
print(y_scaler)

MinMaxScaler()


In [21]:
y_train_scaled = y_scaler.transform(y_train)
print(y_train_scaled)
print(y_train_scaled.shape)

[[0.        ]
 [0.        ]
 [0.01462717]
 ...
 [0.58533333]
 [0.63516667]
 [0.        ]]
(3882, 1)


In [22]:
y_train_scaled_df = pd.DataFrame(y_train_scaled, index=y_train.index, columns=y_train.columns)
y_train_scaled_df

,Generación
30,0.000000
31,0.000000
32,0.014627
33,0.196933
34,0.167691
...,...
12201,0.002433
12202,0.205400
12203,0.585333
12204,0.635167


In [23]:
y_val_scaled = y_scaler.transform(y_val)
print(y_val_scaled)
print(y_val_scaled.shape)

[[0.00000000e+00]
 [1.26666667e-03]
 [8.34000000e-02]
 [5.85333333e-01]
 [6.35166667e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [2.73333333e-03]
 [2.54233333e-01]
 [7.31700000e-01]
 [7.93966667e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [5.70000000e-03]
 [2.57933333e-01]
 [7.31700000e-01]
 [7.93966667e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [2.73333333e-03]
 [2.60933333e-01]
 [7.31700000e-01]
 [1.25666667e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [2.73333333e-03]
 [2.60933333e-01]
 [7.31700000e-01]
 [7.87500000e-01]
 [7.60266667e-01]
 [7.31966667e-01]
 [7.01566667e-01]
 [7.05666667e-01]
 [2.96100000e-01]
 [1.41666667e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [1.10000000e-03]
 [8.35000000e-02]
 [5.85333333e-01]
 [6.27833333e-01]
 [6.08200000e-01]
 [1.13333333e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [1.46666667e-03]
 [8.35000000e-02]
 [5.853333

In [24]:
y_val_scaled_df = pd.DataFrame(y_val_scaled, index=y_val.index, columns=y_val.columns)
y_val_scaled_df

,Generación
12224,0.000000
12225,0.001267
12226,0.083400
12227,0.585333
12228,0.635167
...,...
15897,0.517867
15898,0.829167
15899,0.915400
15908,0.362367


In [25]:
y_test_scaled = y_scaler.transform(y_test)
print(y_test_scaled)
print(y_test_scaled.shape)

[[0.00000000e+00]
 [0.00000000e+00]
 [7.74666667e-02]
 [5.46133333e-01]
 [8.40400000e-01]
 [9.17433333e-01]
 [4.73666667e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [8.25000000e-02]
 [5.39233333e-01]
 [8.24700000e-01]
 [9.11100000e-01]
 [4.57200000e-01]
 [4.96666667e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [7.74666667e-02]
 [5.18166667e-01]
 [8.30233333e-01]
 [9.11033333e-01]
 [4.69000000e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [7.81666667e-02]
 [5.21533333e-01]
 [8.27466667e-01]
 [9.11033333e-01]
 [9.48200000e-01]
 [5.01733333e-01]
 [3.64933333e-01]
 [3.01000000e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [8.21333333e-02]
 [5.18833333e-01]
 [8.27000000e-01]
 [9.11233333e-01]
 [8.55500000e-01]
 [8.21300000e-01]
 [7.80066667e-01]
 [5.63466667e-01]
 [4.56366667e-01]
 [3.40666667e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [4.71666667e-02]
 [3.60533333e-01]
 [5.79133333e-01]
 [5.85166667e-01]
 [6.07233333e-01]
 [6.03966667e-01]
 [5.94433333e-01]
 [5.88066667e-01]
 [5.90700000e-01]
 [4.468666

In [26]:
y_test_scaled_df = pd.DataFrame(y_test_scaled, index=y_test.index, columns=y_test.columns)
y_test_scaled_df

,Generación
15910,0.000000
15919,0.000000
15920,0.077467
15921,0.546133
15922,0.840400
...,...
18273,0.245200
18274,0.587933
18275,0.777967
18276,0.877433


In [27]:
y_scaller_all = MinMaxScaler().fit(y)
print(y_scaller_all)

MinMaxScaler()


In [28]:
y_scaled = y_scaller_all.transform(y)
print(y_scaled)
print(y_scaled.shape)

[[0.        ]
 [0.        ]
 [0.01462717]
 ...
 [0.77796667]
 [0.87743333]
 [0.        ]]
(5546, 1)


In [29]:
y_scaled_df = pd.DataFrame(y_scaled, index=y.index, columns=y.columns)
y_scaled_df

,Generación
30,0.000000
31,0.000000
32,0.014627
33,0.196933
34,0.167691
...,...
18273,0.245200
18274,0.587933
18275,0.777967
18276,0.877433


## Definición de modelos

### RandomForest

In [30]:
from lightgbm import LGBMRegressor
import optuna
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import seaborn as sns
from sklearn.metrics import mean_absolute_percentage_error as mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error as mean_absolute_error
from sklearn.metrics import mean_squared_error as mean_squared_error
from sklearn.metrics import r2_score as r2_score

In [31]:
# Inicializar listas para métricas
LightGBM_model = LGBMRegressor(num_leaves=500, subsample= 0.10698460631792395, colsample_bytree= 0.7272836809565294, min_data_in_leaf= 85)
LightGBM_model.fit(X_train_scaled_df, y_train_scaled_df)
resultados = pd.DataFrame(index = y_test_scaled_df.index, columns=["LightGBM"])
#Ciclo diario de predicción
for i in range(len(X_test)):
    inicio = i * 1
    fin = inicio + 1

    X_test_seg = X_test_scaled_df.iloc[inicio:fin, :]
    y_test_seg = y_test_scaled_df.iloc[inicio:fin]

    if len(X_test_seg) < 1:
        break

    y_pred = LightGBM_model.predict(X_test_seg)
    y_pred = y_scaler.inverse_transform(y_pred.reshape(-1, 1))
    y_pred = np.clip(y_pred, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

    resultados.iloc[i, 0] = y_pred[0, 0]

[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.249167 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 628
[LightGBM] [Info] Number of data points in the train set: 3882, number of used features: 6
[LightGBM] [Info] Start training from score 0.263711
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further split

In [32]:
resultados

,LightGBM
15910,0.0
15919,45.085922
15920,1747.559524
15921,15282.644011
15922,24396.229156
...,...
18273,6918.407231
18274,19779.365821
18275,24030.007366
18276,24112.268947


In [33]:
predicciones = y_test.copy()
predicciones

,Generación
15910,0.0
15919,0.0
15920,2324.0
15921,16384.0
15922,25212.0
...,...
18273,7356.0
18274,17638.0
18275,23339.0
18276,26323.0


In [34]:
predicciones["LightGBM"] = resultados["LightGBM"]
predicciones

,Generación,LightGBM
15910,0.0,0.0
15919,0.0,45.085922
15920,2324.0,1747.559524
15921,16384.0,15282.644011
15922,25212.0,24396.229156
...,...,...
18273,7356.0,6918.407231
18274,17638.0,19779.365821
18275,23339.0,24030.007366
18276,26323.0,24112.268947


In [35]:
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['LightGBM'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['LightGBM']):.4f}")

MAE: 1106.0083
RMSE: 2067.4017
R²: 0.9606


## Random Forest

In [36]:
from sklearn.ensemble import RandomForestRegressor

In [37]:
#Modelo LightGBM
RF_model = RandomForestRegressor(
    criterion="squared_error",
    random_state=0,
    n_estimators=400,
    min_impurity_decrease=0,
    max_depth=None,
    bootstrap=True
)
RF_model.fit(X_train_scaled_df, y_train_scaled_df)
# Inicializar listas para métricas
resultados = pd.DataFrame(index = y_test_scaled_df.index, columns=["Random Forest"])
#Ciclo diario de predicción
for i in range(len(X_test)):
    inicio = i * 1
    fin = inicio + 1

    X_test_seg = X_test_scaled_df.iloc[inicio:fin, :]
    y_test_seg = y_test_scaled_df.iloc[inicio:fin]

    if len(X_test_seg) < 1:
        break

    y_pred = RF_model.predict(X_test_seg)
    y_pred = y_scaler.inverse_transform(y_pred.reshape(-1, 1))
    y_pred = np.clip(y_pred, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

    resultados.iloc[i, 0] = y_pred[0, 0]

In [38]:
predicciones["Random Forest"] = resultados["Random Forest"]
predicciones

,Generación,LightGBM,Random Forest
15910,0.0,0.0,0.0
15919,0.0,45.085922,0.0
15920,2324.0,1747.559524,2931.155236
15921,16384.0,15282.644011,15534.328419
15922,25212.0,24396.229156,25499.346937
...,...,...,...
18273,7356.0,6918.407231,7148.104256
18274,17638.0,19779.365821,21016.223316
18275,23339.0,24030.007366,23687.023299
18276,26323.0,24112.268947,23126.102053


## Preparación redes neuronales

In [39]:
import numpy as np
import pandas as pd

def create_sliding_window_with_index(data_X, data_y, lookback):
    X, y, indices = [], [], []
    
    # Asegurar que `data_y` tiene los mismos índices que `data_X`
    data_y = data_y.reindex(data_X.index)

    max_index = len(data_X) - lookback

    for i in range(max_index):
        X.append(data_X.iloc[i:i + lookback].values)  # Ventana de entrada
        
        # Obtener el índice correcto en `data_y`
        y_index = data_X.index[i + lookback]

        # Extraer el valor correspondiente de `data_y`
        if y_index in data_y.index:
            y_value = data_y.loc[y_index]
            if isinstance(y_value, pd.Series):  # Si devuelve una serie, extraer el valor
                y_value = y_value.iloc[0]
        else:
            y_value = np.nan  # Si no está, asignamos NaN

        y.append(y_value)
        indices.append(y_index)  # 🔹 Guardamos el índice original de `data_y`

    # Convertimos `X` en un array y `y` en DataFrame conservando sus índices originales
    X_array = np.array(X)
    y_df = pd.DataFrame(y, index=indices, columns=['y'])  # 🔹 Conservamos los índices originales

    return X_array, y_df


In [40]:
lookback = 48  # Puedes ajustar a 24, 72, etc.

# Aplicar la ventana deslizante a cada conjunto
X_train_windowed, y_train_windowed = create_sliding_window_with_index(X_train_scaled_df, y_train_scaled_df, lookback)
X_val_windowed, y_val_windowed = create_sliding_window_with_index(X_val_scaled_df, y_val_scaled_df, lookback)
X_test_windowed, y_test_windowed = create_sliding_window_with_index(X_test_scaled_df, y_test_scaled_df, lookback)


In [41]:
print(f'X_train: {X_train_windowed.shape}, y_train: {y_train_windowed.shape}')
print(f'X_val: {X_val_windowed.shape}, y_val: {y_val_windowed.shape}')
print(f'X_test: {X_test_windowed.shape}, y_test: {y_test_windowed.shape}')

X_train: (3834, 48, 6), y_train: (3834, 1)
X_val: (784, 48, 6), y_val: (784, 1)
X_test: (784, 48, 6), y_test: (784, 1)


## CTNET

In [42]:
import tensorflow as tf
from tensorflow.keras import layers

In [43]:
def compile_and_fit(model, xtrain=X_train_windowed, ytrain=y_train_windowed, learning_rate=0.0001):
    model.compile(loss=[tf.keras.losses.MeanSquaredError()],
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=[tf.keras.metrics.RootMeanSquaredError(), tf.keras.metrics.MeanAbsolutePercentageError(), tf.keras.metrics.MeanAbsoluteError()])
    
    history = model.fit(xtrain, ytrain, epochs=50,
                        batch_size=512, validation_split=0.2, verbose=1)
    return history

def Loss(train_loss, valid_loss):
    plt.plot(train_loss)
    plt.plot(valid_loss)
    plt.rcParams["figure.figsize"] = (15, 3)
    plt.title('Model Losses')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train Loss', 'Validation Loss'], loc='upper left')
    plt.savefig('out/loss_plot.png')
    plt.show()

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = layers.LayerNormalization()(inputs)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=128, kernel_size=2, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(norm_x, norm_x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    return norm_x

def build_model(input_shape, head_size, num_heads, ff_dim, num_transformer_blocks, mlp_units, dropout=0, mlp_dropout=0):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs
    
    for _ in range(num_transformer_blocks):
        enc_out = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)
    
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(enc_out, enc_out)
    res = x + enc_out
    x = layers.LayerNormalization(epsilon=1e-6)(res)
    x = layers.GlobalAveragePooling1D(data_format="channels_first")(x)
    x = layers.Dense(832, activation="relu")(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(mlp_dropout)(x)

    outputs = layers.Dense(1)(x)
    
    return tf.keras.Model(inputs, outputs)

In [44]:
CTNET = build_model((X_train_windowed.shape[1], X_train_windowed.shape[2]), head_size=4, num_heads=3, ff_dim=32, num_transformer_blocks=3, mlp_units=[256], mlp_dropout=0.3, dropout=0.2)

In [45]:
history = compile_and_fit(CTNET)

Epoch 1/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 126s 3s/step - loss: 0.1729 - mean_absolute_error: 0.2624 - mean_absolute_percentage_error: 847536.5000 - root_mean_squared_error: 0.4158 - val_loss: 0.1487 - val_mean_absolute_error: 0.2609 - val_mean_absolute_percentage_error: 4711125.5000 - val_root_mean_squared_error: 0.3856
Epoch 2/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 24s 3s/step - loss: 0.1646 - mean_absolute_error: 0.2578 - mean_absolute_percentage_error: 4997073.0000 - root_mean_squared_error: 0.4058 - val_loss: 0.1412 - val_mean_absolute_error: 0.2593 - val_mean_absolute_percentage_error: 10487931.0000 - val_root_mean_squared_error: 0.3757
Epoch 3/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 17s 2s/step - loss: 0.1594 - mean_absolute_error: 0.2575 - mean_absolute_percentage_error: 10245430.0000 - root_mean_squared_error: 0.3993 - val_loss: 0.1325 - val_mean_absolute_error: 0.2580 - val_mean_absolute_percentage_error: 17674666.0000 - val_root_mean_squared_error: 0.3640
Epoch 4/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 24s 2s/step -

In [46]:
CTNET_predictions = CTNET.predict(X_test_windowed)
CTNET_predictions

25/25 ━━━━━━━━━━━━━━━━━━━━ 19s 310ms/step


array([[0.51864487],
       [0.51947576],
       [0.36995572],
       [0.3221997 ],
       [0.31213018],
       [0.2766857 ],
       [0.19010359],
       [0.14884573],
       [0.13346739],
       [0.12699404],
       [0.12500148],
       [0.14970689],
       [0.18471438],
       [0.27267084],
       [0.3285569 ],
       [0.4259632 ],
       [0.52271545],
       [0.5643973 ],
       [0.4861768 ],
       [0.3442145 ],
       [0.24992633],
       [0.24163479],
       [0.209963  ],
       [0.21497734],
       [0.36976975],
       [0.38919804],
       [0.26639885],
       [0.15560883],
       [0.1956476 ],
       [0.28019905],
       [0.27933502],
       [0.30307454],
       [0.44420624],
       [0.4746549 ],
       [0.35517877],
       [0.28594497],
       [0.18787032],
       [0.20198777],
       [0.17934641],
       [0.21745878],
       [0.4035918 ],
       [0.41305402],
       [0.28235206],
       [0.16839628],
       [0.18346736],
       [0.32674587],
       [0.5004322 ],
       [0.519

In [47]:
CTNET_predictions = y_scaler.inverse_transform(CTNET_predictions.reshape(-1, 1))
CTNET_predictions = np.clip(CTNET_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [48]:
resultados = pd.DataFrame(CTNET_predictions, index = y_test_windowed.index, columns=["CTNET"])

In [49]:
predicciones["CTNET"] = resultados["CTNET"]
predicciones

,Generación,LightGBM,Random Forest,CTNET
15910,0.0,0.0,0.0,NaN
15919,0.0,45.085922,0.0,NaN
15920,2324.0,1747.559524,2931.155236,NaN
15921,16384.0,15282.644011,15534.328419,NaN
15922,25212.0,24396.229156,25499.346937,NaN
...,...,...,...,...
18273,7356.0,6918.407231,7148.104256,10369.553711
18274,17638.0,19779.365821,21016.223316,13152.110352
18275,23339.0,24030.007366,23687.023299,14395.314453
18276,26323.0,24112.268947,23126.102053,8805.733398


In [50]:
predicciones["CTNET"] = predicciones["CTNET"].fillna(0)

In [51]:
# import optuna
# import tensorflow as tf
# from tensorflow.keras import layers
# from sklearn.model_selection import train_test_split

# # Definir la función objetivo para Optuna
# def objective(trial):
#     # Sugerir valores para los hiperparámetros
#     head_size = trial.suggest_int("head_size", 8, 64, step=8)
#     num_heads = trial.suggest_int("num_heads", 2, 8, step=2)
#     ff_dim = trial.suggest_int("ff_dim", 32, 256, step=32)
#     num_transformer_blocks = trial.suggest_int("num_transformer_blocks", 1, 4)
#     mlp_units = trial.suggest_categorical("mlp_units", [[128, 64], [256, 128, 64], [512, 256, 128]])
#     dropout = trial.suggest_float("dropout", 0.1, 0.5, step=0.1)
#     mlp_dropout = trial.suggest_float("mlp_dropout", 0.1, 0.5, step=0.1)
#     learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-2)

#     # Construcción del modelo con los hiperparámetros sugeridos
#     model = build_model(
#         input_shape=X_train_windowed.shape[1:],
#         head_size=head_size,
#         num_heads=num_heads,
#         ff_dim=ff_dim,
#         num_transformer_blocks=num_transformer_blocks,
#         mlp_units=mlp_units,
#         dropout=dropout,
#         mlp_dropout=mlp_dropout
#     )

#     # Compilar el modelo con los hiperparámetros sugeridos
#     model.compile(
#         loss=tf.keras.losses.MeanSquaredError(),
#         optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
#         metrics=[tf.keras.metrics.RootMeanSquaredError()]
#     )

#     # Entrenamiento con un número reducido de épocas para acelerar la búsqueda
#     history = model.fit(
#         X_train_windowed, y_train_windowed,
#         validation_split=0.2,
#         epochs=50,  # Reducimos las épocas para acelerar la búsqueda
#         batch_size=512,
#         verbose=0
#     )

#     # Obtener la métrica de validación (RMSE) y minimizarla
#     val_rmse = min(history.history["val_root_mean_squared_error"])
    
#     return val_rmse  # Queremos minimizar el RMSE

# # Ejecutar la optimización de hiperparámetros
# study = optuna.create_study(direction="minimize")
# study.optimize(objective, n_trials=20, timeout=3600)  # 20 iteraciones, máximo 1 hora

# # Mostrar los mejores hiperparámetros encontrados
# best_params = study.best_params
# print(f"Mejores hiperparámetros: {best_params}")


In [52]:
CTNET = build_model((X_train_windowed.shape[1], X_train_windowed.shape[2]), head_size=16, num_heads=8, ff_dim=256, num_transformer_blocks=1, mlp_units=[128,64], mlp_dropout=0.2, dropout=0.2)

In [53]:
history = compile_and_fit(CTNET, learning_rate = 0.0010762230908145116)

Epoch 1/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 184s 5s/step - loss: 0.1653 - mean_absolute_error: 0.2656 - mean_absolute_percentage_error: 7911420.0000 - root_mean_squared_error: 0.4065 - val_loss: 0.0963 - val_mean_absolute_error: 0.2572 - val_mean_absolute_percentage_error: 60133276.0000 - val_root_mean_squared_error: 0.3103
Epoch 2/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 49s 7s/step - loss: 0.1189 - mean_absolute_error: 0.2849 - mean_absolute_percentage_error: 71879704.0000 - root_mean_squared_error: 0.3445 - val_loss: 0.0895 - val_mean_absolute_error: 0.2804 - val_mean_absolute_percentage_error: 121393528.0000 - val_root_mean_squared_error: 0.2992
Epoch 3/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 37s 6s/step - loss: 0.1087 - mean_absolute_error: 0.2899 - mean_absolute_percentage_error: 93315784.0000 - root_mean_squared_error: 0.3297 - val_loss: 0.0886 - val_mean_absolute_error: 0.2619 - val_mean_absolute_percentage_error: 78373688.0000 - val_root_mean_squared_error: 0.2977
Epoch 4/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 46s 6s/st

In [54]:
CTNET_predictions = CTNET.predict(X_test_windowed)
CTNET_predictions

25/25 ━━━━━━━━━━━━━━━━━━━━ 23s 409ms/step


array([[ 7.53708124e-01],
       [ 8.10531199e-01],
       [ 5.62089324e-01],
       [ 8.11576128e-01],
       [ 2.16442049e-01],
       [ 7.95011073e-02],
       [ 4.91415650e-01],
       [ 4.16549116e-01],
       [ 6.67946935e-01],
       [ 2.31645674e-01],
       [-8.43868777e-03],
       [-3.36370245e-03],
       [-1.43427402e-04],
       [ 1.47302840e-02],
       [ 3.61013189e-02],
       [ 2.76758581e-01],
       [ 8.52484524e-01],
       [ 9.86625314e-01],
       [ 1.02069080e-01],
       [ 8.74821842e-03],
       [-4.72890213e-03],
       [-5.26105240e-03],
       [ 5.53645007e-03],
       [ 8.43771547e-02],
       [ 6.32832825e-01],
       [ 1.02343559e+00],
       [ 2.21894234e-02],
       [ 2.56843567e-02],
       [ 2.96005942e-02],
       [ 1.88879892e-02],
       [ 2.62523275e-02],
       [ 2.69189000e-01],
       [ 6.54622912e-01],
       [ 8.65892410e-01],
       [ 9.11053643e-02],
       [ 2.34810919e-01],
       [ 6.65965676e-03],
       [ 1.98730044e-02],
       [ 1.7

In [55]:
CTNET_predictions = y_scaler.inverse_transform(CTNET_predictions.reshape(-1, 1))
CTNET_predictions = np.clip(CTNET_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [56]:
resultados = pd.DataFrame(CTNET_predictions, index = y_test_windowed.index, columns=["CTNET"])

In [57]:
predicciones["CTNET"] = resultados["CTNET"]
predicciones

,Generación,LightGBM,Random Forest,CTNET
15910,0.0,0.0,0.0,NaN
15919,0.0,45.085922,0.0,NaN
15920,2324.0,1747.559524,2931.155236,NaN
15921,16384.0,15282.644011,15534.328419,NaN
15922,25212.0,24396.229156,25499.346937,NaN
...,...,...,...,...
18273,7356.0,6918.407231,7148.104256,8764.482422
18274,17638.0,19779.365821,21016.223316,22835.416016
18275,23339.0,24030.007366,23687.023299,20082.384766
18276,26323.0,24112.268947,23126.102053,555.459778


## Forecasting

In [58]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.metrics import RootMeanSquaredError
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.losses import Huber
from tensorflow.keras.callbacks import EarlyStopping

In [59]:
Forecast_model = Sequential()
Forecast_model.add(InputLayer((X_train_windowed.shape[1], X_train_windowed.shape[2])))

#CNN
Forecast_model.add(Conv1D(filters=64, kernel_size=2, padding='same', activation='relu'))
Forecast_model.add(BatchNormalization())  # 🔹 Nueva Normalización aquí
Forecast_model.add(MaxPooling1D(pool_size=2))

#model_Soleado.add(Flatten())
#BiLSTM
Forecast_model.add(Bidirectional(LSTM(128, return_sequences=True)))
Forecast_model.add(Bidirectional(LSTM(64, return_sequences=True)))
Forecast_model.add(Dropout(0.2))  # 🔹 Mayor regularización en BiLSTM
Forecast_model.add(Bidirectional(LSTM(32, return_sequences=False)))

#Normalización y Dropout
Forecast_model.add(BatchNormalization())
Forecast_model.add(Dropout(0.3))

# Capas Densas
Forecast_model.add(Dense(16, activation='relu'))
Forecast_model.add(Dense(1, 'relu'))

Forecast_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_12 (Conv1D)              │ (None, 48, 64)         │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 48, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 24, 256)        │       197,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 24, 128)        │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 405,601 (1.55 MB)

 Trainable params: 405,345 (1.55 MB)

 Non-trainable params: 256 (1.00 KB)

In [60]:
cp = ModelCheckpoint('Forcasting_model.keras', save_best_only=True)
Forecast_model.compile(optimizer=Adam(learning_rate=0.0001), loss=Huber(delta=1000), metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [61]:
history = Forecast_model.fit(X_train_windowed, y_train_windowed, validation_data=(X_val_windowed, y_val_windowed), epochs=100, batch_size=8, callbacks=[cp, early_stop])

Epoch 1/100
480/480 ━━━━━━━━━━━━━━━━━━━━ 621s 568ms/step - loss: 0.0951 - mae: 0.2865 - val_loss: 0.0955 - val_mae: 0.2996
Epoch 2/100
480/480 ━━━━━━━━━━━━━━━━━━━━ 299s 503ms/step - loss: 0.0844 - mae: 0.2689 - val_loss: 0.0802 - val_mae: 0.2824
Epoch 3/100
480/480 ━━━━━━━━━━━━━━━━━━━━ 258s 482ms/step - loss: 0.0706 - mae: 0.2486 - val_loss: 0.0695 - val_mae: 0.2714
Epoch 4/100
480/480 ━━━━━━━━━━━━━━━━━━━━ 277s 498ms/step - loss: 0.0610 - mae: 0.2284 - val_loss: 0.0642 - val_mae: 0.2531
Epoch 5/100
480/480 ━━━━━━━━━━━━━━━━━━━━ 253s 465ms/step - loss: 0.0550 - mae: 0.2202 - val_loss: 0.0506 - val_mae: 0.2171
Epoch 6/100
480/480 ━━━━━━━━━━━━━━━━━━━━ 251s 429ms/step - loss: 0.0464 - mae: 0.2018 - val_loss: 0.0465 - val_mae: 0.2198
Epoch 7/100
480/480 ━━━━━━━━━━━━━━━━━━━━ 262s 417ms/step - loss: 0.0437 - mae: 0.1908 - val_loss: 0.0406 - val_mae: 0.1949
Epoch 8/100
480/480 ━━━━━━━━━━━━━━━━━━━━ 182s 367ms/step - loss: 0.0368 - mae: 0.1761 - val_loss: 0.0344 - val_mae: 0.1782
Epoch 9/100
480/

In [62]:
Forecast_predictions = Forecast_model.predict(X_test_windowed)
Forecast_predictions

24/25 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step

25/25 ━━━━━━━━━━━━━━━━━━━━ 64s 1s/step 


array([[0.675473  ],
       [0.6923699 ],
       [0.6739727 ],
       [0.66142535],
       [0.6472259 ],
       [0.51986164],
       [0.5738336 ],
       [0.70059645],
       [0.76306796],
       [0.6090367 ],
       [0.3413704 ],
       [0.15909202],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.22674353],
       [0.63749945],
       [0.76000655],
       [0.26309434],
       [0.1428859 ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.21698305],
       [0.6726611 ],
       [0.55174243],
       [0.18023853],
       [0.150624  ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.22060843],
       [0.64515764],
       [0.73153126],
       [0.38481566],
       [0.17520373],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.2770691 ],
       [0.7566146 ],
       [0.6103221 ],
       [0.14495715],
       [0.        ],
       [0.        ],
       [0.30897483],
       [0.8069347 ],
       [0.808

In [63]:
Forecast_predictions = y_scaler.inverse_transform(Forecast_predictions.reshape(-1, 1))
Forecast_predictions = np.clip(Forecast_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [64]:
Forecast_resultados = pd.DataFrame(Forecast_predictions, index = y_test_windowed.index, columns=["Forecast"])

In [65]:
predicciones["Forecast"] = Forecast_resultados["Forecast"]
predicciones

,Generación,LightGBM,Random Forest,CTNET,Forecast
15910,0.0,0.0,0.0,NaN,NaN
15919,0.0,45.085922,0.0,NaN,NaN
15920,2324.0,1747.559524,2931.155236,NaN,NaN
15921,16384.0,15282.644011,15534.328419,NaN,NaN
15922,25212.0,24396.229156,25499.346937,NaN,NaN
...,...,...,...,...,...
18273,7356.0,6918.407231,7148.104256,8764.482422,4792.106445
18274,17638.0,19779.365821,21016.223316,22835.416016,16919.378906
18275,23339.0,24030.007366,23687.023299,20082.384766,20590.408203
18276,26323.0,24112.268947,23126.102053,555.459778,4796.369141


## Métricas

In [66]:
predicciones.loc[~predicciones['CTNET'].isna(),'Generación']

16042    17374.0
16043    17555.0
16044    18217.0
16045    18119.0
16046    17833.0
          ...   
18273     7356.0
18274    17638.0
18275    23339.0
18276    26323.0
18286        0.0
Name: Generación, Length: 784, dtype: float64

## Photovoltaic

In [67]:
from tensorflow.keras.models import Model
inputs = Input(shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]))

# Primera capa CNN
x = Conv1D(filters=64, kernel_size=4, padding='same', activation='relu')(inputs)
x = MaxPooling1D(pool_size=2)(x)

# Segunda capa CNN
x = Conv1D(filters=128, kernel_size=4, padding='same', activation='relu')(x)
x = MaxPooling1D(pool_size=2)(x)

# Capa BiGRU
x = Bidirectional(GRU(64, return_sequences=True))(x)

# Atención: se define de forma explícita
attention = MultiHeadAttention(num_heads=4, key_dim=128)(x, x)

# Aplanar y agregar Dropout
x = Flatten()(attention)
x = Dropout(0.4)(x)
initializer = tf.keras.initializers.HeNormal()
x = Dense(64, activation="relu", kernel_regularizer=l2(0.01))(x)
x = Dense(32, activation="relu")(x)  # Otra capa intermedia

# Capa de salida
outputs = Dense(1, activation="linear")(x)

# Definir el modelo
Photo_model = Model(inputs=inputs, outputs=outputs)

# Resumen del modelo
Photo_model.summary()

Model: "functional_13"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 48, 6)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_13 (Conv1D)  │ (None, 48, 64)    │      1,600 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 24, 64)    │          0 │ conv1d_13[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_14 (Conv1D)  │ (None, 24, 128)   │     32,896 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 12, 128)   │          0 │ conv1d_14[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_3     │ (None, 12, 128)   │     74,496 │ max_pooling1d_2[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 12, 128)   │    263,808 │ bidirectional_3[… │
│ (MultiHeadAttentio… │                   │            │ bidirectional_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 1536)      │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_11          │ (None, 1536)      │          0 │ flatten[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 64)        │     98,368 │ dropout_11[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 32)        │      2,080 │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 1)         │         33 │ dense_11[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 473,281 (1.81 MB)

 Trainable params: 473,281 (1.81 MB)

 Non-trainable params: 0 (0.00 B)

In [68]:
cp2 = ModelCheckpoint('Photovoltaic_model.keras', save_best_only=True)
Photo_model.compile(optimizer=Adam(learning_rate=0.0001), loss="mean_squared_error", metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [69]:
history = Photo_model.fit(
    X_train_windowed, y_train_windowed,
    validation_data=(X_val_windowed, y_val_windowed),
    epochs=50,
    batch_size=16,
    callbacks=[cp, early_stop]
)

Epoch 1/50
240/240 ━━━━━━━━━━━━━━━━━━━━ 321s 278ms/step - loss: 1.0836 - mae: 0.2749 - val_loss: 0.5556 - val_mae: 0.3015
Epoch 2/50
240/240 ━━━━━━━━━━━━━━━━━━━━ 54s 198ms/step - loss: 0.4411 - mae: 0.2683 - val_loss: 0.2421 - val_mae: 0.2438
Epoch 3/50
240/240 ━━━━━━━━━━━━━━━━━━━━ 85s 194ms/step - loss: 0.1782 - mae: 0.1758 - val_loss: 0.1269 - val_mae: 0.2041
Epoch 4/50
240/240 ━━━━━━━━━━━━━━━━━━━━ 81s 180ms/step - loss: 0.0906 - mae: 0.1490 - val_loss: 0.0866 - val_mae: 0.1901
Epoch 5/50
240/240 ━━━━━━━━━━━━━━━━━━━━ 45s 163ms/step - loss: 0.0622 - mae: 0.1425 - val_loss: 0.0750 - val_mae: 0.1828
Epoch 6/50
240/240 ━━━━━━━━━━━━━━━━━━━━ 50s 183ms/step - loss: 0.0478 - mae: 0.1299 - val_loss: 0.0657 - val_mae: 0.1756
Epoch 7/50
240/240 ━━━━━━━━━━━━━━━━━━━━ 84s 173ms/step - loss: 0.0441 - mae: 0.1291 - val_loss: 0.0618 - val_mae: 0.1720
Epoch 8/50
240/240 ━━━━━━━━━━━━━━━━━━━━ 39s 141ms/step - loss: 0.0408 - mae: 0.1256 - val_loss: 0.0567 - val_mae: 0.1637
Epoch 9/50
240/240 ━━━━━━━━━━━━

In [70]:
Photo_predictions = Photo_model.predict(X_test_windowed)
Photo_predictions

25/25 ━━━━━━━━━━━━━━━━━━━━ 44s 853ms/step


array([[ 6.99300587e-01],
       [ 8.07209790e-01],
       [ 6.67223036e-01],
       [ 7.34536111e-01],
       [ 7.64608443e-01],
       [ 6.22417033e-01],
       [ 5.80582619e-01],
       [ 6.99197114e-01],
       [ 7.20320761e-01],
       [ 6.99465811e-01],
       [ 3.96431237e-01],
       [ 2.62478411e-01],
       [ 2.28605196e-02],
       [ 1.31486747e-02],
       [ 1.99984927e-02],
       [ 3.18670839e-01],
       [ 7.41774023e-01],
       [ 8.22249591e-01],
       [ 3.20222348e-01],
       [ 3.06693166e-01],
       [ 1.41302105e-02],
       [-1.95833389e-03],
       [ 8.96332599e-03],
       [ 3.17754418e-01],
       [ 7.32415378e-01],
       [ 7.12480664e-01],
       [ 1.60279363e-01],
       [ 2.02087671e-01],
       [ 1.03992913e-02],
       [-8.84965993e-04],
       [ 1.08201690e-02],
       [ 3.25708300e-01],
       [ 7.29987204e-01],
       [ 7.45371461e-01],
       [ 2.97878087e-01],
       [ 2.91646183e-01],
       [ 2.63324380e-03],
       [ 1.11641269e-02],
       [ 1.4

In [71]:
Photo_predictions = y_scaler.inverse_transform(Photo_predictions.reshape(-1, 1))
Photo_predictions = np.clip(Photo_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [72]:
Photo_resultados = pd.DataFrame(Photo_predictions, index = y_test_windowed.index, columns=["Photo"])

In [73]:
predicciones["Photo"] = Photo_resultados["Photo"]
predicciones

,Generación,LightGBM,Random Forest,CTNET,Forecast,Photo
15910,0.0,0.0,0.0,NaN,NaN,NaN
15919,0.0,45.085922,0.0,NaN,NaN,NaN
15920,2324.0,1747.559524,2931.155236,NaN,NaN,NaN
15921,16384.0,15282.644011,15534.328419,NaN,NaN,NaN
15922,25212.0,24396.229156,25499.346937,NaN,NaN,NaN
...,...,...,...,...,...,...
18273,7356.0,6918.407231,7148.104256,8764.482422,4792.106445,6426.757812
18274,17638.0,19779.365821,21016.223316,22835.416016,16919.378906,19157.732422
18275,23339.0,24030.007366,23687.023299,20082.384766,20590.408203,20639.869141
18276,26323.0,24112.268947,23126.102053,555.459778,4796.369141,7396.311035


In [74]:
print("LightGBM")
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['LightGBM'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print("Random Forest")
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['Random Forest']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['Random Forest'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['Random Forest']):.4f}")
print("CTNET")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET']):.4f}")
print("Forecast")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast']):.4f}")
print("Photovoltaic")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo']):.4f}")

LightGBM
MAE: 1106.0083
RMSE: 2067.4017
R²: 0.9606
Random Forest
MAE: 1090.0346
RMSE: 2258.6311
R²: 0.9530
CTNET
MAE: 3467.4857
RMSE: 5586.7247
R²: 0.7102
Forecast
MAE: 3091.9189
RMSE: 5056.4349
R²: 0.7626
Photovoltaic
MAE: 2914.2414
RMSE: 4709.7532
R²: 0.7940


In [75]:
# Seleccionar las columnas desde "LightGBM" en adelante
columnas_nuevas = predicciones.loc[:, "LightGBM":]

# Unir con `datos` usando el índice, manteniendo todo en `datos`
datos = datos.merge(columnas_nuevas, left_index=True, right_index=True, how='left')

# Ver resultado
datos.head()


,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day,LightGBM,Random Forest,CTNET,Forecast,Photo
24,2022-09-02 00:00:00,0.0,19,76,0,0,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
25,2022-09-02 01:00:00,0.0,18,81,0,1,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
26,2022-09-02 02:00:00,0.0,18,84,0,2,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
27,2022-09-02 03:00:00,0.0,18,86,0,3,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
28,2022-09-02 04:00:00,0.0,17,86,0,4,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN


## X_train para hacer análisis de sobreajuste

In [76]:
predicciones_train = y_train.copy()

In [77]:
LightGBM_predictions_train = LightGBM_model.predict(X_train_scaled_df)
LightGBM_predictions_train = y_scaler.inverse_transform(LightGBM_predictions_train.reshape(-1, 1))
LightGBM_predictions_train = np.clip(LightGBM_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
LightGBM_resultados = pd.DataFrame(LightGBM_predictions_train, index = y_train_scaled_df.index, columns=["LightGBM_train"])
predicciones_train["LightGBM_train"] = LightGBM_resultados["LightGBM_train"]

[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85


In [78]:
RandomForest_predictions_train = RF_model.predict(X_train_scaled_df)
RandomForest_predictions_train = y_scaler.inverse_transform(RandomForest_predictions_train.reshape(-1, 1))
RandomForest_predictions_train = np.clip(RandomForest_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
RandomForest_resultados = pd.DataFrame(RandomForest_predictions_train, index = y_train_scaled_df.index, columns=["RandomForest_train"])
predicciones_train["RandomForest_train"] = RandomForest_resultados["RandomForest_train"]

In [79]:
CTNET_predictions_train = CTNET.predict(X_train_windowed)
CTNET_predictions_train = y_scaler.inverse_transform(CTNET_predictions_train.reshape(-1, 1))
CTNET_predictions_train = np.clip(CTNET_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
CTNET_resultados = pd.DataFrame(CTNET_predictions_train, index = y_train_windowed.index, columns=["CTNET_train"])
predicciones_train["CTNET_train"] = CTNET_resultados["CTNET_train"]

120/120 ━━━━━━━━━━━━━━━━━━━━ 21s 95ms/step


In [80]:
Forecast_predictions_train = Forecast_model.predict(X_train_windowed)
Forecast_predictions_train = y_scaler.inverse_transform(Forecast_predictions_train.reshape(-1, 1))
Forecast_predictions_train = np.clip(Forecast_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
Forecast_resultados = pd.DataFrame(Forecast_predictions_train, index = y_train_windowed.index, columns=["Forecast_train"])
predicciones_train["Forecast_train"] = Forecast_resultados["Forecast_train"]

120/120 ━━━━━━━━━━━━━━━━━━━━ 22s 137ms/step


In [81]:
Photo_predictions_train = Photo_model.predict(X_train_windowed)
Photo_predictions_train = y_scaler.inverse_transform(Photo_predictions_train.reshape(-1, 1))
Photo_predictions_train = np.clip(Photo_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
Photo_resultados = pd.DataFrame(Photo_predictions_train, index = y_train_windowed.index, columns=["Photo_train"])
predicciones_train["Photo_train"] = Photo_resultados["Photo_train"]

120/120 ━━━━━━━━━━━━━━━━━━━━ 13s 62ms/step


In [82]:
predicciones_train

,Generación,LightGBM_train,RandomForest_train,CTNET_train,Forecast_train,Photo_train
30,0.000000,60.077787,0.000000,NaN,NaN,NaN
31,0.000000,125.231595,27.698762,NaN,NaN,NaN
32,438.814997,884.746075,725.392330,NaN,NaN,NaN
33,5908.000884,10270.994439,6697.509600,NaN,NaN,NaN
34,5030.740421,8168.208553,7058.407504,NaN,NaN,NaN
...,...,...,...,...,...,...
12201,73.000000,21.688955,78.895340,1237.663940,0.000000,2421.334961
12202,6162.000000,4498.638662,5359.067161,6596.325195,7734.854980,8587.843750
12203,17560.000000,11523.395944,16822.837755,16539.724609,10022.355469,10730.244141
12204,19055.000000,18153.461478,18481.512500,18394.330078,17319.433594,15284.959961


In [83]:
print("LightGBM")
print(f"MAE: {mean_absolute_error(predicciones_train['Generación'], predicciones_train['LightGBM_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train['Generación'], predicciones_train['LightGBM_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train['Generación'], predicciones_train['LightGBM_train']):.4f}")
print("Random Forest")
print(f"MAE: {mean_absolute_error(predicciones_train['Generación'], predicciones_train['RandomForest_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train['Generación'], predicciones_train['RandomForest_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train['Generación'], predicciones_train['RandomForest_train']):.4f}")
print("CTNET")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train']):.4f}")
print("Forecast")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train']):.4f}")
print("Photovoltaic")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train']):.4f}")

LightGBM
MAE: 1250.4265
RMSE: 2294.1852
R²: 0.9420
Random Forest
MAE: 499.6361
RMSE: 1002.8449
R²: 0.9889
CTNET
MAE: 2138.8703
RMSE: 3548.0421
R²: 0.8617
Forecast
MAE: 3760.0102
RMSE: 6126.0682
R²: 0.5876
Photovoltaic
MAE: 3246.3386
RMSE: 4868.2346
R²: 0.7396


In [84]:
# Seleccionar las columnas desde "LightGBM" en adelante
columnas_nuevas = predicciones_train.loc[:, "LightGBM_train":]

# Unir con `datos` usando el índice, manteniendo todo en `datos`
datos = datos.merge(columnas_nuevas, left_index=True, right_index=True, how='left')

# Ver resultado
datos.head()


,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day,LightGBM,Random Forest,CTNET,Forecast,Photo,LightGBM_train,RandomForest_train,CTNET_train,Forecast_train,Photo_train
24,2022-09-02 00:00:00,0.0,19,76,0,0,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,2022-09-02 01:00:00,0.0,18,81,0,1,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26,2022-09-02 02:00:00,0.0,18,84,0,2,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2022-09-02 03:00:00,0.0,18,86,0,3,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28,2022-09-02 04:00:00,0.0,17,86,0,4,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [85]:
datos.to_excel("04.8_Predicciones_Conjunto_soleado KMeans CTNET.xlsx", index=True)

## Guardamos los modelos

In [86]:
import joblib

# Guardar modelo LightGBM
joblib.dump(LightGBM_model, "4_8_LightGBM_model.pkl")

# Guardar modelo Random Forest
joblib.dump(RF_model, "4_8_RandomForest_model.pkl")


['4_8_RandomForest_model.pkl']

In [87]:
CTNET.save("4_8_CTNET_model.keras")
Forecast_model.save("4_8_Forecast_model.keras")
Photo_model.save("4_8_Photo_model.keras")